# GNN Baseline Replication

This notebook tests paper-style GNN baselines for AML edge classification on HI-Small with a 60/20/20 temporal split and dynamic snapshot protocol. It trains PNA-like and GIN+EU-like models with imbalance-aware loss and evaluates minority-class detection performance. The results are intended as a practical baseline comparison against the tree-based and MEGA-style notebooks.

In [1]:
import json
import math
import platform
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # PyG install (best effort for Colab CUDA runtime)
    %pip -q install -U pip
    %pip -q install scikit-learn
    TORCH = torch.__version__.split('+')[0]
    CUDA = 'cpu' if torch.version.cuda is None else f"cu{torch.version.cuda.replace('.', '')}"
    WHEEL_URL = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"
    for pkg in ['pyg-lib', 'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg, '-f', WHEEL_URL], check=False)
    %pip -q install torch-geometric

from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import PNAConv, GINEConv

# Verify PyG sampling backend availability (required by LinkNeighborLoader at runtime).
try:
    import pyg_lib  # noqa: F401
    HAS_PYG_LIB = True
except Exception:
    HAS_PYG_LIB = False

try:
    import torch_sparse  # noqa: F401
    HAS_TORCH_SPARSE = True
except Exception:
    HAS_TORCH_SPARSE = False

USE_NEIGHBOR_SAMPLING = HAS_PYG_LIB or HAS_TORCH_SPARSE

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('HAS_PYG_LIB:', HAS_PYG_LIB, 'HAS_TORCH_SPARSE:', HAS_TORCH_SPARSE)
print('USE_NEIGHBOR_SAMPLING:', USE_NEIGHBOR_SAMPLING)
if not USE_NEIGHBOR_SAMPLING:
    print('Warning: LinkNeighborLoader backend missing. Falling back to full-graph training path.')

PROJECT_ROOT = Path('/content/drive/MyDrive/math-168') if IS_COLAB else Path('/Users/sophiasharif/projects/math-168')
DATA_DIR = PROJECT_ROOT / 'data'
TRANS_CSV = DATA_DIR / 'HI-Small_Trans.csv'
ACCOUNTS_CSV = DATA_DIR / 'HI-Small_accounts.csv'
if not TRANS_CSV.exists() or not ACCOUNTS_CSV.exists():
    raise FileNotFoundError(f'Missing HI-Small files in {DATA_DIR}')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.12
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
PROJECT_ROOT: /content/drive/MyDrive/math-168
DATA_DIR: /content/drive/MyDrive/math-168/data


In [2]:
# Data prep
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def normalize_bank(s):
    return s.astype(str).str.strip().str.lstrip('0').replace('', '0')

trans = pd.read_csv(TRANS_CSV)
accounts = pd.read_csv(ACCOUNTS_CSV, usecols=['Bank ID', 'Account Number'])

src_bank = normalize_bank(trans['From Bank'])
src_acct = trans['Account'].astype(str).str.strip()
dst_bank = normalize_bank(trans['To Bank'])
dst_acct = trans['Account.1'].astype(str).str.strip()

df_all = pd.DataFrame({
    'src': src_bank + '_' + src_acct,
    'dst': dst_bank + '_' + dst_acct,
    'timestamp': pd.to_datetime(trans['Timestamp'], errors='coerce'),
    'label': pd.to_numeric(trans['Is Laundering'], errors='coerce').fillna(0).astype(int).clip(0, 1),
    'amount_received': pd.to_numeric(trans['Amount Received'], errors='coerce').fillna(0.0),
    'amount_paid': pd.to_numeric(trans['Amount Paid'], errors='coerce').fillna(0.0),
    'receiving_currency_code': trans['Receiving Currency'].astype('category').cat.codes,
    'payment_currency_code': trans['Payment Currency'].astype('category').cat.codes,
    'payment_format_code': trans['Payment Format'].astype('category').cat.codes,
}).dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

acct_bank = normalize_bank(accounts['Bank ID'])
acct_num = accounts['Account Number'].astype(str).str.strip()
acct_keys = set((acct_bank + '_' + acct_num).tolist())
print('src coverage:', round(df_all['src'].isin(acct_keys).mean(), 4))
print('dst coverage:', round(df_all['dst'].isin(acct_keys).mean(), 4))
print('full rows:', len(df_all), 'positives:', int(df_all['label'].sum()))

# For practicality in Colab; increase if your runtime handles it.
MAX_EDGES = 1_500_000
df = df_all.iloc[:min(MAX_EDGES, len(df_all))].copy().reset_index(drop=True)

nodes = pd.Index(df['src']).append(pd.Index(df['dst'])).unique().tolist()
node_to_idx = {n: i for i, n in enumerate(nodes)}
src = df['src'].map(node_to_idx).astype(np.int64).values
dst = df['dst'].map(node_to_idx).astype(np.int64).values
y = df['label'].astype(np.float32).values

edge_attr = df[[
    'amount_received',
    'amount_paid',
    'receiving_currency_code',
    'payment_currency_code',
    'payment_format_code',
]].values.astype(np.float32)
edge_attr = (edge_attr - edge_attr.mean(0, keepdims=True)) / (edge_attr.std(0, keepdims=True) + 1e-8)

n = len(df)
tr, va = int(0.6 * n), int(0.8 * n)
train_ids = np.arange(0, tr)
val_ids = np.arange(tr, va)
test_ids = np.arange(va, n)

print('subset rows:', n, 'positives:', int(y.sum()))
print('split positives train/val/test:', int(y[train_ids].sum()), int(y[val_ids].sum()), int(y[test_ids].sum()))
print('num_nodes:', len(nodes))

src coverage: 1.0
dst coverage: 1.0
full rows: 5078345 positives: 5177
subset rows: 1500000 positives: 502
split positives train/val/test: 237 124 141
num_nodes: 474155


In [3]:
# Build temporal snapshots and neighbor loaders
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

edge_index_np = np.vstack([src, dst])
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
edge_attr_t = torch.tensor(edge_attr, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

def make_snapshot_graph(edge_end_idx):
    # Graph snapshot includes edges [0, edge_end_idx)
    ei = edge_index[:, :edge_end_idx]
    ea = edge_attr_t[:edge_end_idx]
    x = torch.ones((len(nodes), 1), dtype=torch.float32)
    return Data(x=x, edge_index=ei, edge_attr=ea)

train_graph = make_snapshot_graph(tr)
val_graph = make_snapshot_graph(va)
test_graph = make_snapshot_graph(n)

def make_loader(graph, edge_ids, batch_size=4096, shuffle=False):
    edge_label_index = edge_index[:, edge_ids]
    edge_label = y_t[edge_ids]
    return LinkNeighborLoader(
        data=graph,
        num_neighbors=[100, 100],  # paper setting
        batch_size=batch_size,
        edge_label_index=edge_label_index,
        edge_label=edge_label,
        shuffle=shuffle,
    )

if USE_NEIGHBOR_SAMPLING:
    train_loader = make_loader(train_graph, train_ids, batch_size=4096, shuffle=True)
    val_loader = make_loader(val_graph, val_ids, batch_size=4096, shuffle=False)
    test_loader = make_loader(test_graph, test_ids, batch_size=4096, shuffle=False)
else:
    train_loader = val_loader = test_loader = None

print('Graph snapshots ready on device:', device, '| neighbor_sampling=', USE_NEIGHBOR_SAMPLING)

/usr/local/lib/python3.12/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


Loaders ready on device: cuda


In [4]:
# Models: PNA-like and GIN+EU-like
class PNAEdgeModel(nn.Module):
    def __init__(self, edge_dim, hidden=64, layers=2, dropout=0.2, deg=None):
        super().__init__()
        self.dropout = dropout
        self.node_proj = nn.Linear(1, hidden)
        self.convs = nn.ModuleList()
        for _ in range(layers):
            self.convs.append(
                PNAConv(
                    in_channels=hidden,
                    out_channels=hidden,
                    aggregators=['mean', 'min', 'max', 'std'],
                    scalers=['identity', 'amplification', 'attenuation'],
                    deg=deg,
                    edge_dim=edge_dim,
                )
            )
        self.edge_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, batch):
        x, ei, ea = batch.x, batch.edge_index, batch.edge_attr
        h = self.node_proj(x)
        for conv in self.convs:
            h = conv(h, ei, ea)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        src, dst = batch.edge_label_index
        out = self.edge_head(torch.cat([h[src], h[dst]], dim=-1)).squeeze(-1)
        return out

class GINEEUModel(nn.Module):
    def __init__(self, edge_dim, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_proj = nn.Linear(1, hidden)
        self.edge_proj = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList()
        self.edge_updates = nn.ModuleList()
        for _ in range(layers):
            nn_mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(nn_mlp, edge_dim=hidden))
            self.edge_updates.append(nn.Sequential(nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Linear(hidden, hidden)))
        self.edge_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, batch):
        x, ei, ea = batch.x, batch.edge_index, batch.edge_attr
        h = self.node_proj(x)
        e = self.edge_proj(ea)
        for conv, eup in zip(self.convs, self.edge_updates):
            h = conv(h, ei, e)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            src_e, dst_e = ei[0], ei[1]
            e = eup(torch.cat([h[src_e], h[dst_e], e], dim=-1))
        src, dst = batch.edge_label_index
        out = self.edge_head(torch.cat([h[src], h[dst]], dim=-1)).squeeze(-1)
        return out

def f1_best_threshold(y_true, p):
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.01, 0.99, 0.01):
        f = f1_score(y_true, (p >= t).astype(int), zero_division=0)
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t, float(best_f)

def eval_loader(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            p = torch.sigmoid(logits).detach().cpu().numpy()
            yb = batch.edge_label.detach().cpu().numpy()
            probs.append(p)
            labels.append(yb)
    return np.concatenate(labels), np.concatenate(probs)

def eval_full_graph(model, graph, edge_ids):
    from types import SimpleNamespace
    model.eval()
    with torch.no_grad():
        batch = SimpleNamespace(
            x=graph.x.to(device),
            edge_index=graph.edge_index.to(device),
            edge_attr=graph.edge_attr.to(device),
            edge_label_index=edge_index[:, edge_ids].to(device),
            edge_label=y_t[edge_ids].to(device),
        )
        logits = model(batch)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        yb = batch.edge_label.detach().cpu().numpy()
    return yb, p

def train_one(model_name='pna', minority_weight=8.0, epochs=8, lr=1e-3, dropout=0.2):
    # degree histogram for PNA scalers
    deg = torch.bincount(train_graph.edge_index[1], minlength=train_graph.num_nodes)
    if model_name == 'pna':
        model = PNAEdgeModel(edge_dim=edge_attr.shape[1], hidden=64, layers=2, dropout=dropout, deg=deg).to(device)
    else:
        model = GINEEUModel(edge_dim=edge_attr.shape[1], hidden=64, layers=2, dropout=dropout).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos_weight = torch.tensor(float(minority_weight), dtype=torch.float32, device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_state, best_val_f1, best_thr = None, -1.0, 0.5
    stale, patience = 0, 3
    for ep in range(1, epochs + 1):
        model.train()
        losses = []

        if USE_NEIGHBOR_SAMPLING:
            for batch in train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                logits = model(batch)
                loss = crit(logits, batch.edge_label.float())
                loss.backward()
                opt.step()
                losses.append(float(loss.item()))
            yv, pv = eval_loader(model, val_loader)
        else:
            # Fallback when pyg-lib/torch-sparse backend is unavailable.
            from types import SimpleNamespace
            train_batch = SimpleNamespace(
                x=train_graph.x.to(device),
                edge_index=train_graph.edge_index.to(device),
                edge_attr=train_graph.edge_attr.to(device),
                edge_label_index=edge_index[:, train_ids].to(device),
                edge_label=y_t[train_ids].to(device),
            )
            opt.zero_grad()
            logits = model(train_batch)
            loss = crit(logits, train_batch.edge_label.float())
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
            yv, pv = eval_full_graph(model, val_graph, val_ids)

        thr, vf1 = f1_best_threshold(yv, pv)
        print(f'[{model_name}] epoch={ep} loss={np.mean(losses):.4f} val_f1={vf1:.4f} thr={thr:.2f}')
        if vf1 > best_val_f1:
            best_val_f1, best_thr = vf1, thr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break

    model.load_state_dict(best_state)
    if USE_NEIGHBOR_SAMPLING:
        yt, pt = eval_loader(model, test_loader)
    else:
        yt, pt = eval_full_graph(model, test_graph, test_ids)
    pred = (pt >= best_thr).astype(int)
    return {
        'model': model_name,
        'minority_weight': float(minority_weight),
        'lr': float(lr),
        'dropout': float(dropout),
        'best_val_f1': float(best_val_f1),
        'best_threshold': float(best_thr),
        'test_f1': float(f1_score(yt, pred, zero_division=0)),
        'test_precision': float(precision_score(yt, pred, zero_division=0)),
        'test_recall': float(recall_score(yt, pred, zero_division=0)),
        'test_pr_auc': float(average_precision_score(yt, pt)),
    }

# Small paper-aligned tuning grid
results = []
for w in [6.0, 8.0]:
    results.append(train_one(model_name='pna', minority_weight=w, epochs=8, lr=1e-3, dropout=0.2))
for w in [6.0, 8.0]:
    # GIN range in paper uses higher LR than PNA; use conservative lower end to stabilize.
    results.append(train_one(model_name='gine_eu', minority_weight=w, epochs=8, lr=5e-3, dropout=0.2))

results_sorted = sorted(results, key=lambda r: r['best_val_f1'], reverse=True)
print('\nTop result:')
print(json.dumps(results_sorted[0], indent=2))

artifacts = PROJECT_ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)
with open(artifacts / 'gnn_baseline_replication_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results_sorted, f, indent=2)

print('\nSaved:', artifacts / 'gnn_baseline_replication_metrics.json')
print('Paper HI-Small references (Table 2): PNA ~0.5677, GIN+EU ~0.4773, GIN ~0.2870')

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'